# Step1 导入相关模块


In [17]:
# md 文档编辑好后esc - m - shift enter

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,DataCollatorForSeq2Seq, TrainingArguments,Trainer, AutoModel

In [18]:
import datasets
datasets.__version__

'4.8.5'

In [8]:
import transformers
transformers.__version__


'5.3.0'

In [5]:
import warnings
warnings.filterwarnings('ignore')

# Step2 加载数据集

In [19]:
ds = load_dataset('json', data_dir="../alpaca_data_zh/")
ds = ds['train'] # 加载数据集后放到train数据集中
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 48818
})

In [11]:
# 使用切面查看部分样本数据
ds[:2]

{'instruction': ['保持健康的三个提示。', '三原色是什么？'],
 'input': ['', ''],
 'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。']}

# Step3 数据预处理

In [12]:
import requests

print(requests.get("https://huggingface.co").status_code)

200


In [20]:
# 分词器,langbot
# AutoTokenizer 读取指定模型中文件
# 需要梯子后才能访问下载huggle face

model_name = 'Langboat/bloom-1b4-zh'

model_path_local = r"D:\models\bloom-1b4-zh"


tokenizer = AutoTokenizer.from_pretrained(model_path_local)
tokenizer

TokenizersBackend(name_or_path='D:\models\bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [21]:
# 使用分词器读取内容并处理样本
def process_func(example):
    # 单条样本最大长度
    MAX_LENGTH = 256
    # 将文本数据交给模型后转为模型能看得懂的数据，模型关注id;transformers相关的大模型都要计算【注意力机制】，所以有attention_mask；
    input_ids, attention_mask, labels = [], [], []
    # 使用tokenizer做分词, 告诉模型你作为Assistant开始给我回答
    instruction = tokenizer("\n".join(["Human:"+example['instruction'], example['input']]).strip() + "\n\nAssistant:")
    # eos_token为结束符，表示这句话说完了
    response = tokenizer(example['output'] + tokenizer.eos_token)

    input_ids = instruction['input_ids'] + response['input_ids']
    attention_mask = instruction['attention_mask'] + response['attention_mask']
    # 只关心后半段 response['input_ids']
    labels = [-100] * len(instruction['input_ids']) + response['input_ids']

    if len(input_ids)> MAX_LENGTH:
        # 大于长度后，对数据做切片操作
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return{
        'input_ids':input_ids, 'attention_mask':attention_mask, 'labels':labels
    }
    

In [22]:
# 调用函数处理每条样本数据
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Map: 100%|██████████| 48818/48818 [00:23<00:00, 2054.02 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 48818
})

In [23]:
# 截取部分数据查看
tokenized_ds[:2]

# 解码
tokenizer.decode(tokenized_ds[:2]['input_ids'])

['Human:保持健康的三个提示。\n\nAssistant:以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。</s>',
 'Human:三原色是什么？\n\nAssistant:三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。</s>']

In [ ]:
# 解码 labels中不等于 -100 的内容
tokenizer.decode(list(filter(lambda x: x!=-100, tokenized_ds[2]['labels'])))

# Step4 模型创建

In [36]:
# 导入模型
# model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage = True)
# model

# 导入本地模型
model = AutoModelForCausalLM.from_pretrained(model_path_local)

Loading weights: 100%|██████████| 294/294 [00:00<00:00, 31396.41it/s]
The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# 当前什么设备
model.device()

# 汇总参数量
sum(param.numel() for param in model.parameters())

# 根据参数量计算需要的显存

# float32 用4个字节存储
model size: 1.36 * 4 ≈ 5.26
# 做全量参数调参
SGD gradient ： 1.36 * 4 ≈ 5.26
with Momentum Optimizer : 1.36 * 4 ≈ 5.26
with Adam Optimizer: 还有计算二阶动量 : 1.36 * 4 ≈ 5.26

# 总共
1.36 * 4 * 4 ≈ 20.8G

# Prompt-Tunning

## PEFT Step1 配置文件

In [ ]:
from peft import PromptTuningConfig, get_peft_model, TaskType, PromptTuningInit

In [34]:
# soft prompt
# config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens= 10)
# config

# hard prompt
config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="下面是一段与人的对话",
    num_virtual_tokens=len(tokenizer("下面是一段与人的对话")['input_ids']),
    tokenizer_name_or_path=model_path_local
)
config

PromptTuningConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.PROMPT_TUNING: 'PROMPT_TUNING'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, num_virtual_tokens=6, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, modules_to_save=None, prompt_tuning_init=<PromptTuningInit.TEXT: 'TEXT'>, prompt_tuning_init_text='下面是一段与人的对话', tokenizer_name_or_path='D:\\models\\bloom-1b4-zh', tokenizer_kwargs=None)

## PEFT Step2 创建模型

In [38]:
print(type(model))
print(model is not None)

model = get_peft_model(model, config)
model

<class 'transformers.models.bloom.modeling_bloom.BloomForCausalLM'>
True


PeftModelForCausalLM(
  (base_model): BloomForCausalLM(
    (transformer): BloomModel(
      (word_embeddings): Embedding(46145, 2048)
      (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (h): ModuleList(
        (0-23): 24 x BloomBlock(
          (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (self_attention): BloomAttention(
            (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
            (dense): Linear(in_features=2048, out_features=2048, bias=True)
            (attention_dropout): Dropout(p=0.0, inplace=False)
          )
          (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (mlp): BloomMLP(
            (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
            (gelu_impl): BloomGelu()
            (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
          )
        )
      )

In [39]:
# 可训练的参数量
model.print_trainable_parameters()

trainable params: 12,288 || all params: 1,397,628,928 || trainable%: 0.0009


# Step5 配置训练参数

In [40]:
args = TrainingArguments(
    output_dir='./chatbot', # 输出文件夹，存储模型预测结果和模型文件checkpoints
    per_device_train_batch_size=1, # 训练模型时，每个GPU核/CPU 上面对应的batchSize大小（样本数）
    gradient_accumulation_steps=8, # 执行反向传播/更新参数前，对应梯度计算累计了 n 次
    logging_steps=10, # 每隔 n 次迭代，落地一次日志
    num_train_epochs=1 # 模型学习数据集学习 n 遍
)


# Step6 创建训练器

In [41]:
# 获取训练器
trainer = Trainer(
    model= model, # 虽然传的对象，但部分参数已经被冻结了
    args= args,
    train_dataset= tokenized_ds,
    data_collator= DataCollatorForSeq2Seq(tokenizer=tokenizer, padding = True), # 构建一批次数据
)

In [42]:
# 开始训练
trainer.train()

Step,Training Loss
10,2.382144
20,2.383723
30,2.393056
40,2.361418
50,2.220067
60,2.348193
70,2.471265
80,2.345909
90,2.273002
100,2.324866


KeyboardInterrupt: 

# 加载训练好的PEFT模型

In [43]:
from peft import PeftModel

# 传入的model内容为原始模型,训练好的向量
peft_model = PeftModel.from_pretrained(model=model, model_id = './chatbot/checkpoint-1500')

# STEP8 模型推理

In [44]:
peft_model.device
# 如果使用的cpu进行训练，可指定使用gpu
peft_model = model.cuda()

In [ ]:
ipt = tokenizer("Human:{}\n{}". format("如何提高学习效率？","").strip() + "\n\nAssistant:", 
                return_tensors = 'pt').to(model.device)
# model输出转为文本
out = tokenizer.decode(peft_model.generate(**ipt, max_length = 256, do_sample=True)[0], skip_special_tokens=True)

print(out)

'Human:如何提高学习效率？\n\nAssistant:在面对大量的信息时，需要不断去思考和总结，这样一来往往会导致学习的效率下降。\n\n为了提高学习效率，你可以试着将注意力集中在最重要的信息上。学习时，你可以利用时间管理等工具，避免在学习到一半的时候中断。你可以运用多种方式提高学习效率，例如尝试快速阅读和思考。\n\n想要提高学习效率，你也可以尝试减少时间焦虑。学习时经常与他人进行交流，这样可以让你更快的了解别人的想法，并学会从他人的观点中更好地吸收知识。此外，你也可以尝试改变学习地点，并减少学习时间。'